In [ ]:
# ==============================================================================
# CureSense — Kaggle Notebook (Flask pipeline only)
# main.ipynb
#
# Runs: Whisper + GLiNER + RAG + PDF analysis + Interview pipeline
# GPU:  T4 ×2  (Whisper + GLiNER on GPU 0, GPU 1 unused)
# URL:  AI_SERVICE_URL in Backend/.env
#
# CT/MRI and X-ray analysis (MedGemma) runs in a SEPARATE Google Colab
# notebook — see colab_medgemma.ipynb. That URL goes into MEDGEMMA_SERVICE_URL.
#
# Cell order (run top-to-bottom on first use):
#   1  Pull code from GitHub
#   2  Install dependencies
#   3  Secrets + OpenAI client
#   4  Load Whisper + GLiNER models
#   5  Load or build dual RAG index (clinician + patient)   ← skip on re-runs if already built
#   5b Load or build disease index                          ← skip on re-runs if already built
#   7  Load both indexes into memory                        ← always run before Flask
#   8  Start Flask on port 5001
#   9  ngrok tunnel → Flask:5001
#
# On subsequent Kaggle sessions: skip Cells 5 and 5b (files already in working
# dir or Kaggle dataset). Always run Cell 7 — it loads indexes into memory fresh
# each session. Flask (Cell 8) must start after Cell 7.
# ==============================================================================

In [ ]:
# ── Cell 1: Pull code from GitHub ────────────────────────────────────────────
import sys, os, subprocess

REPO_URL = 'https://github.com/moizaimran/curesense-project.git'
BRANCH   = 'hassan-branch'
REPO_DIR = '/kaggle/working/curesense-project'

if not os.path.exists(REPO_DIR):
    print('Cloning repo ...')
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    print('Repo already cloned — pulling latest ...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

sys.path.insert(0, REPO_DIR)
print('Ready:', REPO_DIR)

In [ ]:
# ── Cell 2: Installs ──────────────────────────────────────────────────────────
# requirements.txt covers Flask service deps (pdfplumber, pypdfium2, openai,
# whisper, gliner, faiss, etc.)
import sys
REPO_DIR = '/kaggle/working/curesense-project'
!pip install -r {REPO_DIR}/requirements.txt -q

# pyngrok — ngrok tunnel in Cell 9
!pip install -q pyngrok

!pip install torchvision --upgrade --quiet

# rapidfuzz — fuzzy disease-name matching for ICD-10 lookup in build_disease_index
!pip install rapidfuzz -q


In [ ]:
# ── Cell 3: Secrets + OpenAI client ──────────────────────────────────────────
#
# Kaggle secrets required (Settings → Secrets → Add new secret):
#   'Ngrok Key'   — ngrok auth token
#   'OpenAI Key'  — OpenAI API key
#
# NOTE: HF_TOKEN is NOT needed here — MedGemma runs in the Colab notebook.
#
from kaggle_secrets import UserSecretsClient
from pyngrok import ngrok
from openai import OpenAI
import glinker.config as cfg

_secrets = UserSecretsClient()

ngrok.set_auth_token(_secrets.get_secret('Ngrok Key'))
cfg.openai_client = OpenAI(api_key=_secrets.get_secret('OpenAI Key'))

print('OpenAI client ready')
print('ngrok ready')

In [ ]:
# ── Cell 4: Load heavy models (Whisper + GLiNER) on GPU 0 ────────────────────
import torch
import whisper
from gliner import GLiNER
import glinker.config as cfg

print('Loading Whisper Large ...')
cfg.whisper_model = whisper.load_model('large', device='cuda')
print('Whisper ready')

print('Loading GLiNER BioMed ...')
cfg.gliner_model = GLiNER.from_pretrained('Ihor/gliner-biomed-bi-large-v1.0').to('cuda')
print('GLiNER ready on:', next(cfg.gliner_model.parameters()).device)

In [ ]:
# ── Cell 5: Load or build the RAG index ──────────────────────────────────────
#
# Priority order:
#   1. /kaggle/working/rag_index/                                    — already built this session
#   2. /kaggle/input/datasets/hassanraheem/uresense-rag-index/       — pre-built dataset
#   3. Build from scratch                                             — first time only (~30-60 min)
#
# Files expected (dual-index — audience-split):
#   clinician_index.faiss + clinician_meta.json   (textbooks + clinician/mixed guidelines)
#   patient_index.faiss   + patient_meta.json     (MedRAG/wikidoc + patient/mixed guidelines)
#
# On subsequent runs, skip this cell — Cell 7 loads the index into memory.
#
import os, shutil, glob
from glinker.rag.ingestion import build_index
import glinker.config as cfg

WORKING_INDEX = cfg.RAG_INDEX_DIR
DATASET_MOUNT = '/kaggle/input/datasets/hassanraheem/uresense-rag-index'

if os.path.exists(f"{WORKING_INDEX}/clinician_index.faiss"):
    print("Dual RAG index already in working dir — skipping build/copy. Cell 7 will load.")
else:
    matches = glob.glob(f"{DATASET_MOUNT}/**/clinician_index.faiss", recursive=True)
    if not matches:
        matches = glob.glob(f"{DATASET_MOUNT}/clinician_index.faiss")

    if matches:
        source_dir = os.path.dirname(matches[0])
        print(f"Pre-built dataset found at: {source_dir} — copying to working dir ...")
        os.makedirs(WORKING_INDEX, exist_ok=True)
        for fname in ("clinician_index.faiss", "clinician_meta.json",
                      "patient_index.faiss",   "patient_meta.json"):
            src = f"{source_dir}/{fname}"
            if os.path.exists(src):
                shutil.copy(src, f"{WORKING_INDEX}/{fname}")
                print(f"  copied {fname}")
            else:
                print(f"  WARNING: {fname} not found in dataset — will degrade gracefully")
        print("Done. Cell 7 will load both indexes into memory.")
    else:
        print(f"clinician_index.faiss not found under {DATASET_MOUNT}")
        print(f"Contents: {os.listdir(DATASET_MOUNT) if os.path.exists(DATASET_MOUNT) else 'dataset not mounted'}")
        print("Building from scratch (~30-60 min) ...")
        build_index(textbooks_limit=5000)   # guidelines_limit and wikidoc_limit default to None (full corpus)
        print("Index built and saved to", WORKING_INDEX)
        print("Cell 7 will load into memory.")

In [ ]:
# ── Cell 5b: Load or build the Disease Index ────────────────────────────────
#
# Same priority pattern as Cell 5 (RAG index):
#   1. /kaggle/working/disease_index/    — already built this session
#   2. Pre-built Kaggle dataset          — upload once, reuse every session
#   3. Build from scratch                — first time only (~20-40 min)
#
# FIRST TIME ONLY: run this cell once, download the output files
# (disease_index.faiss + disease_meta.json), upload them as a Kaggle dataset
# named  curesense-disease-index  so future sessions skip the build step.
#
import os, shutil, glob
import glinker.config as cfg

WORKING_DISEASE  = '/kaggle/working/disease_index'
DATASET_DISEASE  = '/kaggle/input/datasets/hassanraheem/curesense-disease-index'

if os.path.exists(f"{WORKING_DISEASE}/disease_index.faiss"):
    print("Disease index already in working dir — skipping.")
else:
    # Try to copy from pre-built Kaggle dataset first (fast — avoids 20-min rebuild)
    matches = glob.glob(f"{DATASET_DISEASE}/**/disease_index.faiss", recursive=True)
    if not matches:
        matches = glob.glob(f"{DATASET_DISEASE}/disease_index.faiss")

    if matches:
        source_dir = os.path.dirname(matches[0])
        print(f"Pre-built disease dataset found at: {source_dir} — copying ...")
        os.makedirs(WORKING_DISEASE, exist_ok=True)
        for fname in ("disease_index.faiss", "disease_meta.json"):
            shutil.copy(f"{source_dir}/{fname}", f"{WORKING_DISEASE}/{fname}")
        print("Copied. Cell 6 will load it.")
    else:
        print("No pre-built disease index found — building from scratch (~20-40 min).")
        print("After this completes, download the output files and upload as a Kaggle dataset")
        print("named  curesense-disease-index  to skip this step in future sessions.")
        from scripts.build_disease_index import build_disease_index
        build_disease_index()


In [ ]:
# ── Cell 5b: Load or build the Disease Index ────────────────────────────────
#
# Same priority pattern as Cell 5 (RAG index):
#   1. /kaggle/working/disease_index/    — already built this session
#   2. Pre-built Kaggle dataset          — upload once, reuse every session
#   3. Build from scratch                — first time only (~5-8 min)
#
# FIRST TIME ONLY: run this cell once, download the output files
# (disease_index.faiss + disease_meta.json), upload them as a Kaggle dataset
# named  curesense-disease-index  so future sessions skip the build step.
#
import os, shutil, glob

WORKING_DISEASE  = '/kaggle/working/disease_index'
DATASET_DISEASE  = '/kaggle/input/datasets/hassanraheem/curesense-disease-index'

if os.path.exists(f"{WORKING_DISEASE}/disease_index.faiss"):
    print("Disease index already in working dir — skipping.")
else:
    matches = glob.glob(f"{DATASET_DISEASE}/**/disease_index.faiss", recursive=True)
    if not matches:
        matches = glob.glob(f"{DATASET_DISEASE}/disease_index.faiss")

    if matches:
        source_dir = os.path.dirname(matches[0])
        print(f"Pre-built disease dataset found at: {source_dir} — copying ...")
        os.makedirs(WORKING_DISEASE, exist_ok=True)
        for fname in ("disease_index.faiss", "disease_meta.json"):
            shutil.copy(f"{source_dir}/{fname}", f"{WORKING_DISEASE}/{fname}")
        print("Copied. Cell 6 will load it.")
    else:
        print("No pre-built disease index found — building from scratch (~5-8 min).")
        print("After this completes, download the output files and upload as a Kaggle dataset")
        print("named  curesense-disease-index  to skip this step in future sessions.")
        from scripts.build_disease_index import build_disease_index
        build_disease_index()

In [ ]:
# ── Cell 7: Load RAG + Disease indexes into memory ───────────────────────────
#
# Must run AFTER Cell 5 (RAG files) and Cell 5b (Disease files),
# and BEFORE Cell 8 (Flask startup).
#
# Populates the module-level state that retrieve_context(),
# retrieve_patient_context(), and retrieve_diseases() read at request time.
# Flask serves requests immediately on startup — if this cell is skipped,
# all RAG retrieval silently returns [] for the entire session.
#
from glinker.rag.retrieval import load_index
from glinker.disease.disease_retrieval import load_disease_index

load_index()          # clinician_index.faiss + patient_index.faiss → memory
load_disease_index()  # disease_index.faiss → memory

In [ ]:
# ── Cell 8: Start Flask on port 5001 ─────────────────────────────────────────
import threading
from api.app import app as flask_app

_flask_thread = threading.Thread(
    target=lambda: flask_app.run(port=5001, debug=False, use_reloader=False),
    daemon=True,
)
_flask_thread.start()
print("Flask started on port 5001")

In [ ]:
# ── Cell 9: ngrok tunnel → Flask:5001 ────────────────────────────────────────
#
# Exposes the Flask service (PDF analysis, interview pipeline, Whisper, GLiNER)
# at a public HTTPS URL. Use this URL as AI_SERVICE_URL in Backend/.env.
#
# MedGemma (CT/MRI + X-ray) has its OWN URL from the Colab notebook.
# That URL goes into MEDGEMMA_SERVICE_URL in Backend/.env.
#
import time, socket
from pyngrok import ngrok

def _port_in_use(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("localhost", port)) == 0

# Wait for Flask to be fully ready
for _ in range(15):
    if _port_in_use(5001):
        break
    time.sleep(1)

# Disconnect any stale tunnels from previous runs
for t in ngrok.get_tunnels():
    print(f"[ngrok] Disconnecting old tunnel: {t.public_url}")
    ngrok.disconnect(t.public_url)

PUBLIC_URL = ngrok.connect(5001).public_url
print(f"[ngrok] Connected → {PUBLIC_URL} → Flask:5001")

print("\n" + "=" * 64)
print(f"  AI_SERVICE_URL = {PUBLIC_URL}")
print("=" * 64)
print("\nPaste into Backend/.env as AI_SERVICE_URL and restart Express.")
print("Also run the Colab notebook and paste its URL as MEDGEMMA_SERVICE_URL.")